# Ontario Hospital Analytics Pipeline

This notebook, I ingest 4 hospital datasets, clean each dataframe, build a hospital dimension table, join the fact tables to it, and write the result to an SQLite file.

In [ ]:
from pathlib import Path
import re
import sqlite3

import pandas as pd

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "Data"
DB_PATH = PROJECT_DIR / "ontario_hospitals.db"

DATA_FILES = {
    "ed_wait_times": "hqo_ed_waittimes.csv",
    "safety_hospitals": "hqo_patient_safety_hospitals.csv",
    "safety_timeseries": "hqo_patient_safety_timeseries.csv",
    "geography": "odhf_ontario_hospitals.csv",
}

pd.set_option("display.max_columns", 50)

In [ ]:
def clean_column_names(df):
    cleaned = df.copy()
    cleaned.columns = (
        cleaned.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return cleaned


def normalize_hospital_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()
    name = re.sub(r"\s*[-\u2013\u2014]\s*(site|campus|location|hospital|health|centre|center).*$", "", name)
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    return re.sub(r"\s+", " ", name).strip()


def coerce_numeric(df, columns):
    cleaned = df.copy()
    for column in columns:
        if column in cleaned.columns:
            cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")
    return cleaned


def add_name_key(df, name_column="hospital_name"):
    keyed = df.copy()
    keyed["name_key"] = keyed[name_column].map(normalize_hospital_name)
    return keyed.drop_duplicates(subset="name_key").reset_index(drop=True)

## Load Raw Data

In [ ]:
raw_ed_wait_times = pd.read_csv(DATA_DIR / DATA_FILES["ed_wait_times"])
raw_safety_hospitals = pd.read_csv(DATA_DIR / DATA_FILES["safety_hospitals"])
raw_safety_timeseries = pd.read_csv(DATA_DIR / DATA_FILES["safety_timeseries"])
raw_geography = pd.read_csv(DATA_DIR / DATA_FILES["geography"])

raw_shapes = pd.DataFrame(
    [
        ("ed_wait_times", *raw_ed_wait_times.shape),
        ("safety_hospitals", *raw_safety_hospitals.shape),
        ("safety_timeseries", *raw_safety_timeseries.shape),
        ("geography", *raw_geography.shape),
    ],
    columns=["dataset", "rows", "columns"],
)
raw_shapes

,dataset,rows,columns
0,ed_wait_times,167,5
1,safety_hospitals,224,5
2,safety_timeseries,52,16
3,geography,239,10


## Clean ED Wait Times

In [ ]:
ed_wait_times = clean_column_names(raw_ed_wait_times).rename(
    columns={
        "admitted_patients": "wait_admitted_hrs",
        "first_assessment_by_a_doctor": "wait_first_assess_hrs",
        "high_urgency_patients_not_admitted": "wait_high_urgency_hrs",
        "low_urgency_patients_not_admitted": "wait_low_urgency_hrs",
    }
)

ed_wait_columns = [
    "wait_admitted_hrs",
    "wait_first_assess_hrs",
    "wait_high_urgency_hrs",
    "wait_low_urgency_hrs",
]

ed_wait_times = (
    ed_wait_times
    .pipe(coerce_numeric, ed_wait_columns)
    .pipe(add_name_key)
)

ed_wait_times.head()

,hospital_name,wait_admitted_hrs,wait_first_assess_hrs,wait_high_urgency_hrs,wait_low_urgency_hrs,name_key
0,Alexandra Marine and General Hospital,8.3,1.8,3.8,3.1,alexandra marine and general hospital
1,Almonte General Hospital,7.1,2.7,4.0,3.5,almonte general hospital
2,Anson General Hospital,9.0,1.2,3.8,2.7,anson general hospital
3,Arnprior Regional Health,8.6,2.4,4.7,3.8,arnprior regional health
4,Atikokan General Hospital,5.8,0.6,2.0,1.4,atikokan general hospital


## Clean Patient Safety by Hospital

In [ ]:
patient_safety = clean_column_names(raw_safety_hospitals).rename(
    columns={"hand_hygiene_pct_before": "hand_hygiene_pct"}
)

safety_metric_columns = [
    "antibiotic_resistant_rate",
    "cdiff_rate",
    "hand_hygiene_pct",
    "surgical_safety_pct",
]

patient_safety = patient_safety.pipe(coerce_numeric, safety_metric_columns)
patient_safety["name_key"] = patient_safety["hospital_name"].map(normalize_hospital_name)

# Keep one row per hospital, preferring rows with the most populated safety metrics.
patient_safety = (
    patient_safety
    .assign(metric_count=patient_safety[safety_metric_columns].notna().sum(axis=1))
    .sort_values(["name_key", "metric_count"], ascending=[True, False])
    .drop_duplicates(subset="name_key", keep="first")
    .drop(columns="metric_count")
    .reset_index(drop=True)
)

# Preserve missingness, then create imputed columns for analysis.
for column in safety_metric_columns:
    patient_safety[f"{column}_missing"] = patient_safety[column].isna()
    patient_safety[f"{column}_imputed"] = patient_safety[column].fillna(
        patient_safety[column].median()
    )

patient_safety.head()

,hospital_name,antibiotic_resistant_rate,cdiff_rate,hand_hygiene_pct,surgical_safety_pct,name_key,antibiotic_resistant_rate_missing,antibiotic_resistant_rate_imputed,cdiff_rate_missing,cdiff_rate_imputed,hand_hygiene_pct_missing,hand_hygiene_pct_imputed,surgical_safety_pct_missing,surgical_safety_pct_imputed
0,Alexandra Hospital,0.0,NaN,NaN,NaN,alexandra hospital,False,0.0,True,0.065121,True,86.03,True,100.0
1,Alexandra Marine and General Hospital,NaN,NaN,NaN,NaN,alexandra marine and general hospital,True,0.0,True,0.065121,True,86.03,True,100.0
2,Almonte General Hospital,NaN,NaN,66.49,NaN,almonte general hospital,True,0.0,True,0.065121,False,66.49,True,100.0
3,Anson General Hospital,0.0,0.0,NaN,NaN,anson general hospital,False,0.0,False,0.000000,True,86.03,True,100.0
4,Arnprior Regional Health,0.0,0.0,NaN,100.0,arnprior regional health,False,0.0,False,0.000000,True,86.03,False,100.0


In [ ]:
patient_safety.shape

(224, 6)

## Clean Patient Safety Time Series

In [ ]:
safety_timeseries = clean_column_names(raw_safety_timeseries)
safety_timeseries = safety_timeseries[
    ["indicator", "orgname", "rate", "cases", "key1", "key2"]
].rename(
    columns={
        "orgname": "org_name",
        "key1": "quarter_key",
        "key2": "quarter_label",
    }
)

safety_timeseries = coerce_numeric(safety_timeseries, ["rate", "cases"])
safety_timeseries = safety_timeseries.query("org_name == 'Ontario'").reset_index(drop=True)

safety_timeseries.head()

,indicator,org_name,rate,cases,quarter_key,quarter_label
0,Antibiotic-Resistant Bloodstream Infections,Ontario,0.025263,72.0,2023Q1,"Apr 01-Jun 30, 2023"
1,Antibiotic-Resistant Bloodstream Infections,Ontario,0.029524,79.0,2023Q2,"Jul 01-Sep 30, 2023"
2,Antibiotic-Resistant Bloodstream Infections,Ontario,0.042543,118.0,2023Q3,"Oct 01-Dec 31, 2023"
3,Antibiotic-Resistant Bloodstream Infections,Ontario,0.037791,96.0,2023Q4,"Jan 01-Mar 31, 2024"
4,Antibiotic-Resistant Bloodstream Infections,Ontario,0.037071,108.0,2024Q1,"Apr 01-Jun 30, 2024"


## Clean Hospital Geography

In [ ]:
hospital_geography = clean_column_names(raw_geography).rename(
    columns={
        "facility_name": "hospital_name",
        "csdname": "municipality",
    }
)

hospital_geography = hospital_geography.query("province == 'on'").copy()
hospital_geography = hospital_geography[
    ["hospital_name", "city", "postal_code", "municipality", "latitude", "longitude"]
]

hospital_geography["city"] = hospital_geography["city"].str.title()
hospital_geography["postal_code"] = hospital_geography["postal_code"].str.upper().str.strip()
hospital_geography = (
    hospital_geography
    .pipe(coerce_numeric, ["latitude", "longitude"])
    .pipe(add_name_key)
)

hospital_geography.head()

,hospital_name,city,postal_code,municipality,latitude,longitude,name_key
0,Alexandra Hospital,Ingersoll,N5C 3V6,Ingersoll,43.032068,-80.875459,alexandra hospital
1,Alexandra Marine and General Hospital,Goderich,N7A 1W5,Goderich,43.750293,-81.706020,alexandra marine and general hospital
2,Alexandra Marine and General Hospital of Goderich,Goderich,N7A 3A2,Goderich,43.735844,-81.698324,alexandra marine and general hospital of goderich
3,Algoma Family Services,Sault Ste Marie,P6B 1Y3,Sault Ste. Marie,46.521541,-84.317832,algoma family services
4,Almonte General Hospital,Almonte,K0A 1A0,Mississippi Mills,45.229134,-76.189495,almonte general hospital


## Build Hospital Dimension

In [ ]:
known_hospitals = pd.concat(
    [
        ed_wait_times[["hospital_name", "name_key"]],
        patient_safety[["hospital_name", "name_key"]],
    ],
    ignore_index=True,
).drop_duplicates(subset="name_key")

missing_from_geography = known_hospitals.loc[
    ~known_hospitals["name_key"].isin(hospital_geography["name_key"])
].copy()

for column in ["city", "postal_code", "municipality", "latitude", "longitude"]:
    missing_from_geography[column] = pd.NA

dim_hospitals = pd.concat(
    [hospital_geography, missing_from_geography[hospital_geography.columns]],
    ignore_index=True,
).sort_values("hospital_name", kind="stable")

dim_hospitals = dim_hospitals.reset_index(drop=True)
dim_hospitals.insert(0, "hospital_id", dim_hospitals.index + 1)

dim_hospitals.head()

/tmp/ipykernel_23281/1971506583.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dim_hospitals = pd.concat(


,hospital_id,hospital_name,city,postal_code,municipality,latitude,longitude,name_key
0,1,Alexandra Hospital,Ingersoll,N5C 3V6,Ingersoll,43.032068,-80.875459,alexandra hospital
1,2,Alexandra Marine and General Hospital,Goderich,N7A 1W5,Goderich,43.750293,-81.706020,alexandra marine and general hospital
2,3,Alexandra Marine and General Hospital of Goderich,Goderich,N7A 3A2,Goderich,43.735844,-81.698324,alexandra marine and general hospital of goderich
3,4,Algoma Family Services,Sault Ste Marie,P6B 1Y3,Sault Ste. Marie,46.521541,-84.317832,algoma family services
4,5,Almonte General Hospital,Almonte,K0A 1A0,Mississippi Mills,45.229134,-76.189495,almonte general hospital


## Join Facts to Dimension

In [ ]:
hospital_ids = dim_hospitals.set_index("name_key")["hospital_id"]

fact_ed_wait_times = ed_wait_times.assign(
    hospital_id=ed_wait_times["name_key"].map(hospital_ids)
).drop(columns="name_key")

fact_patient_safety = patient_safety.assign(
    hospital_id=patient_safety["name_key"].map(hospital_ids)
).drop(columns="name_key")

dim_hospitals = dim_hospitals.drop(columns="name_key")

match_summary = pd.DataFrame(
    [
        ("ed_wait_times", fact_ed_wait_times["hospital_id"].notna().sum(), len(fact_ed_wait_times)),
        ("patient_safety", fact_patient_safety["hospital_id"].notna().sum(), len(fact_patient_safety)),
    ],
    columns=["table", "matched_rows", "total_rows"],
)
match_summary

,table,matched_rows,total_rows
0,ed_wait_times,167,167
1,patient_safety,224,224


## Write SQLite Tables

In [ ]:
tables = {
    "dim_hospitals": dim_hospitals,
    "fact_ed_waittimes": fact_ed_wait_times,
    "fact_patient_safety": fact_patient_safety,
    "fact_safety_timeseries": safety_timeseries,
}

with sqlite3.connect(DB_PATH) as connection:
    for table_name, dataframe in tables.items():
        dataframe.to_sql(table_name, connection, if_exists="replace", index=False)

DB_PATH

PosixPath('/content/ontario_hospitals.db')

## Sanity Check

In [ ]:
with sqlite3.connect(DB_PATH) as connection:
    table_counts = pd.read_sql(
        """
        SELECT 'dim_hospitals' AS table_name, COUNT(*) AS rows FROM dim_hospitals
        UNION ALL SELECT 'fact_ed_waittimes', COUNT(*) FROM fact_ed_waittimes
        UNION ALL SELECT 'fact_patient_safety', COUNT(*) FROM fact_patient_safety
        UNION ALL SELECT 'fact_safety_timeseries', COUNT(*) FROM fact_safety_timeseries
        """,
        connection,
    )

table_counts

,table_name,rows
0,dim_hospitals,422
1,fact_ed_waittimes,167
2,fact_patient_safety,224
3,fact_safety_timeseries,52
